# Tez Görselleri (Thesis Visuals)
Bu notebook, tezde kullanılacak 3.x bölümlerindeki eksik görselleştirmeleri üretir.


In [1]:
import sys
from pathlib import Path
sys.path.append('../src')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.fftpack
try:
    import pywt
except ImportError:
    print("Hata: 'pywt' yüklü değil. 'pip install PyWavelets' ile yükleyin.")

plt.rcParams.update({
    'font.family':        'DejaVu Serif',
    'font.size':          11,
    'axes.grid':          True,
    'grid.alpha':         0.3,
    'figure.dpi':         150,
})


## 3.2.1 CUSUM ve 3.3 / 3.4 Çıktıları
Bu bölümlerin görselleri (CUSUM, Piecewise RUL, MI Skorları) zaten diğer notebooklarda (02_cusum_analysis, 03_comparison_plots) oluşturulduğundan, bu notebookta tekrar edilmemiştir. Sadece henüz üretilmemiş olan eksik görseller yer almaktadır.


## 3.2.3 Özellik Çıkarımı (Feature Extraction) Subplot
Bir pencerenin Ham Sinyal → Wavelet Denoising → FFT Spektrumu haritasını çıkarır.


In [2]:
# Bir örnek sinyal yükleyelim (Bearing1_1, dosya 50)
DATA_ROOT = Path('../data/raw/Learning_set')
path = DATA_ROOT / 'Bearing1_1' / 'acc_00050.csv'
if path.exists():
    df = pd.read_csv(path, header=None, names=['h', 'm', 's', 'us', 'h_acc', 'v_acc'])
    sig = df['h_acc'].values
    fs = 25600
    t = np.arange(len(sig)) / fs * 1000 # ms
    
    # 1. Raw
    raw_sig = sig
    
    # 2. Wavelet Denoising (Soft Thresholding)
    if 'pywt' in sys.modules:
        wavelet = 'db4'
        level = 3
        coeffs = pywt.wavedec(raw_sig, wavelet, level=level)
        sigma = np.median(np.abs(coeffs[-1])) / 0.6745
        uthresh = sigma * np.sqrt(2 * np.log(len(raw_sig)))
        coeffs_denoised = [pywt.threshold(c, value=uthresh, mode='soft') for c in coeffs]
        denoised_sig = pywt.waverec(coeffs_denoised, wavelet)
        denoised_sig = denoised_sig[:len(raw_sig)]
    else:
        denoised_sig = raw_sig # Fallback
        print("Denoising atlandı.")
        
    # 3. FFT of Denoised
    yf = scipy.fftpack.fft(denoised_sig)
    xf = np.linspace(0.0, fs/2.0, len(denoised_sig)//2)
    mag = 2.0/len(denoised_sig) * np.abs(yf[:len(denoised_sig)//2])
    
    # Plotting
    fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
    
    axes[0].plot(t, raw_sig, color="#607D8B", lw=0.8)
    axes[0].set_title("1. Ham Sinyal (Raw Signal)")
    axes[0].set_xlabel("Zaman (ms)")
    axes[0].set_ylabel("İvme (g)")
    axes[0].set_xlim([0, t[-1]])
    
    axes[1].plot(t, denoised_sig, color="#4CAF50", lw=0.8)
    axes[1].set_title("2. Wavelet Denoising (db4)")
    axes[1].set_xlabel("Zaman (ms)")
    axes[1].set_xlim([0, t[-1]])
    
    mask = xf <= 5000
    axes[2].plot(xf[mask], mag[mask], color="#9C27B0", lw=1)
    axes[2].set_title("3. Frekans Spektrumu (FFT)")
    axes[2].set_xlabel("Frekans (Hz)")
    
    plt.tight_layout()
    out_dir = Path('../experiments/results')
    out_dir.mkdir(parents=True, exist_ok=True)
    plt.savefig(out_dir / 'feature_extraction_pipeline.png', bbox_inches='tight')
    plt.show()
else:
    print('Veri dosyası bulunamadı, yol: ', path)


ValueError: buffer source array is read-only

## 3.5.2 PHM Score Eğrisi
`evaluation.py` içerisinde bulunan fonksiyonla asimetrik ceza eğrisinin çizilmesi.


In [ ]:
from evaluation import plot_phm_scoring_function

out_dir = Path('../experiments/results')
out_dir.mkdir(parents=True, exist_ok=True)
save_path = out_dir / 'phm_scoring_function.png'

plot_phm_scoring_function(save_path=save_path)
print(f'PHM Skoru görseli kaydedildi: {save_path}')
